> **中文阅读版**：本文件是 `13_runtime_safety_showcase.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 运行时安全展示：过期数据、kill switch、对账、健康状态

**Docker 镜像**：`ml4t`

**章节参考**：25.7（运维就绪）

**实现技能**：
- `ml4t.live.safety`：强制执行 `max_data_staleness_seconds` 和 `max_daily_loss`
- `ml4t.live.safety`：持久化 `RiskState` 与 `SafeBroker.connect()` 启动对账
- `ml4t.live.engine`：`LiveEngine.runtime_status()` 以及 `ok` / `waiting_for_data` /
  `feed_silent` / `idle_market_closed` / `broker_disconnected` / `stopped` 健康状态
- `ml4t-live` CLI 作为进程外操作界面

**本 Notebook 存在的原因**

`10_safety_risk_demo` 会逐一介绍可配置的风险面，即策略作者向操作员暴露的设置。本 Notebook 演练运行时信任契约：在实盘部署必须存活的那类场景中，*故障情况下*会发生什么。只有当控制措施在其防护条件出现时实际触发，预检清单才有用。这里，每个控制措施都被驱动到其设计针对的故障模式。

**学习目标**
- 观察 `SafeBroker` 拒绝针对过期 `MarketSnapshot` 的订单以及针对日亏损
  突破的订单，并确认 kill switch 在 `SafeBroker` 重建后仍保持锁存。
- 检查来自故意偏离的持久化状态文件的非干净 `reconciliation_report`，然后解决它并重新连接以获得干净报告。
- 读取 `LiveEngine.runtime_status()`，并观察引擎在 §25.7 所述健康
  状态之间转换。
- 使用 `ml4t-live` CLI 在进程外检查持久化状态文件。

**前置条件**
- `ml4t-live` 已按 `pyproject.toml` 锁定的版本安装（在仓库根目录执行 `uv sync`）。
- 阅读 §25.7 了解运维框架；阅读 `10_safety_risk_demo` 了解可配置风险面。
- 无经纪商凭据且无交易所访问权限：每个演示都针对合成经纪商运行。

## 设置

导入、日志记录，以及一个满足 `SafeBroker` 所期望协议的最小异步经纪商。该合成经纪商被有意设计为有状态，以便每个演示都能塑造其持仓、订单和账户价值快照，而无需涉及实盘交易场所。

In [1]:
"""Runtime Safety Showcase: stale-data, daily-loss, reconciliation, and engine health states."""

import asyncio
import json
import logging
import os
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import polars as pl
from ml4t.backtest.strategy import Strategy
from ml4t.backtest.types import Order, OrderSide, OrderStatus, OrderType, Position
from ml4t.live import LiveRiskConfig, RiskLimitError, RiskState, SafeBroker
from ml4t.live.engine import LiveEngine

from utils.paths import get_output_dir

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
logger = logging.getLogger(__name__)


def run_demo(awaitable):
    """Run an async demo outside the notebook kernel's active event loop."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(lambda: asyncio.run(awaitable)).result()

In [2]:
# Production defaults; papermill may inject overrides
STALE_DATA_WAIT_SECONDS = 1.5  # how long to sleep before retrying with stale data
KILL_SWITCH_LOSS_USD = 1_000.0  # synthetic equity drawdown driving the trip
HEALTH_OBSERVATION_SECONDS = 6  # total wall time spent observing engine health states
STATE_DIR = get_output_dir(25, "runtime_safety_showcase") / "temporary_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)

### 合成经纪商

查询接口仅暴露 `SafeBroker` 所需的状态。订单提交放在单独单元格中，以便读者区分经纪商观察与经纪商变更。

In [3]:
class DemoBrokerQueries:
    """Connection and read-only methods required by `SafeBroker`."""

    async def connect(self) -> None:
        self._connected = True

    async def disconnect(self) -> None:
        self._connected = False

    async def is_connected_async(self) -> bool:
        return self._connected

    @property
    def execution_capabilities(self):
        return frozenset()

    def assert_paper_trading(self) -> None:
        return None

    def get_position(self, asset: str) -> Position | None:
        return self.positions.get(asset)

    async def get_position_async(self, asset: str) -> Position | None:
        return self.positions.get(asset)

    async def get_positions_async(self) -> dict[str, Position]:
        return dict(self.positions)

    async def get_pending_orders_async(self) -> list[Order]:
        return list(self.pending_orders)

    async def get_account_value_async(self) -> float:
        return self.account_value

    async def get_cash_async(self) -> float:
        return self.account_value

该可变经纪商在本地追加合成订单，且从不建立外部连接。

In [4]:
class DemoBroker(DemoBrokerQueries):
    def __init__(
        self,
        *,
        positions: dict[str, Position] | None = None,
        pending_orders: list[Order] | None = None,
        account_value: float = 100_000.0,
    ) -> None:
        self._connected = False
        self.positions = dict(positions or {})
        self.pending_orders = list(pending_orders or [])
        self.account_value = float(account_value)
        self._order_counter = 0

    async def submit_order_async(
        self,
        asset: str,
        quantity: float,
        side: OrderSide | None = None,
        order_type: OrderType = OrderType.MARKET,
        limit_price: float | None = None,
        stop_price: float | None = None,
        **_: Any,
    ) -> Order:
        if side is None:
            side = OrderSide.BUY if quantity > 0 else OrderSide.SELL
            quantity = abs(quantity)
        self._order_counter += 1
        order = Order(
            asset=asset,
            side=side,
            quantity=quantity,
            order_type=order_type,
            limit_price=limit_price,
            stop_price=stop_price,
            order_id=f"DEMO-{self._order_counter:04d}",
            status=OrderStatus.PENDING,
            created_at=datetime.now(UTC),
        )
        self.pending_orders.append(order)
        return order

    async def cancel_order_async(self, order_id: str) -> bool:
        return False

    async def close_position_async(self, asset: str) -> Order | None:
        return None

## 1. 过期数据拒绝

每当引擎接收 bar 时，`SafeBroker` 会按资产保存一个 `MarketSnapshot`。每个
订单意图都会根据该快照的年龄和
`LiveRiskConfig.max_data_staleness_seconds` 进行检查。针对早于该
阈值的快照的订单会抛出 `RiskLimitError`，因为最新已知价格已不再足够可信
以用于计算仓位规模。

下面的演示将过期阈值设为 1 秒，接受一笔基于新鲜
快照的订单，然后等待超过该阈值并提交另一笔订单。第二笔订单必须
失败。这里不重试，也不回退到过期价格：策略会等待新鲜数据
或停止操作。

In [5]:
def _temp_state_path(prefix: str) -> Path:
    """Return a non-existent path inside the isolated notebook output directory."""
    fd, name = tempfile.mkstemp(prefix=prefix, suffix=".json", dir=STATE_DIR)
    os.close(fd)
    p = Path(name)
    p.unlink(missing_ok=True)
    return p


def _cleanup_state_path(path: Path) -> None:
    """Remove a temporary risk-state file and its default sibling journal."""
    journal = path.with_name(f"{path.stem}-journal{path.suffix or '.json'}l")
    for candidate in (
        path,
        path.with_name(f"{path.name}.lock"),
        journal,
        journal.with_name(f"{journal.name}.lock"),
        journal.with_name(f"{journal.name}.head"),
    ):
        candidate.unlink(missing_ok=True)


stale_state = _temp_state_path("nb13_stale_")
stale_broker = DemoBroker()
stale_safe = SafeBroker(
    stale_broker,
    LiveRiskConfig(
        execution_mode="paper",
        max_order_value=10_000.0,
        max_data_staleness_seconds=1.0,
        state_file=str(stale_state),
    ),
)
run_demo(stale_safe.connect())

# Fresh snapshot, order accepted
stale_safe._record_market_data(datetime.now(UTC), {"DEMO": {"close": 100.0}}, {})
fresh_order = run_demo(stale_safe.submit_order_async("DEMO", 10))
print(f"fresh_data_order: accepted ({fresh_order.order_type.value} {fresh_order.quantity} DEMO)")

# Wait past staleness threshold
time.sleep(STALE_DATA_WAIT_SECONDS)

try:
    run_demo(stale_safe.submit_order_async("DEMO", 1))
except RiskLimitError as exc:
    print(f"stale_data_block: {exc}")
else:
    raise AssertionError("Stale market data did not block the second order")

run_demo(stale_safe.disconnect())
_cleanup_state_path(stale_state)

fresh_data_order: accepted (market 10.0 DEMO)


stale_data_block: Stale market data for DEMO: 1.5s old exceeds max 1.0s


第二笔订单从未到达经纪商适配器，且 `RiskLimitError` 同时携带
测得的年龄和配置的阈值，这正是操作员在
放宽过期窗口、排查数据馈送和停止交易之间做决策所需要的。

交易场所中断期间数据馈送静默是常见的实盘故障，而在
中断期间按最后已知价格交易，正会让止损策略变成其反面。
过期检查是对 §25.7 预检要求——价格必须最新而非过期——的运行时强制执行。

## 2. 日亏损突破时自动触发 kill switch

每次订单提交还会读取经纪商的账户价值。第一次读取会锚定
会话的起始权益；之后每次读取都会以相对该锚点的跌幅来衡量日内亏损，
并以零为下限。当亏损超过 `LiveRiskConfig.max_daily_loss` 时，kill
switch 锁存，订单被拒绝。

该演示通过在会话中途变更经纪商的 `account_value` 来模拟权益回撤。
第一笔订单锚定会话起点；权益下降；下一笔订单触发
kill switch。持久化的 `RiskState` 记录激活原因，因此当日重启
会以锁存仍有效的状态恢复。

In [6]:
killswitch_state = _temp_state_path("nb13_killswitch_")
killswitch_broker = DemoBroker(account_value=100_000.0)
killswitch_safe = SafeBroker(
    killswitch_broker,
    LiveRiskConfig(
        execution_mode="paper",
        max_order_value=10_000.0,
        max_daily_loss=KILL_SWITCH_LOSS_USD,
        max_data_staleness_seconds=60.0,
        state_file=str(killswitch_state),
    ),
)
run_demo(killswitch_safe.connect())
killswitch_safe._record_market_data(datetime.now(UTC), {"DEMO": {"close": 100.0}}, {})

# Anchor session_start_equity at $100k
run_demo(killswitch_safe.submit_order_async("DEMO", 1))
print(f"pre_drawdown_state: kill_switch={killswitch_safe._state.kill_switch_activated}")

# Simulated equity drop
killswitch_broker.account_value = 100_000.0 - (KILL_SWITCH_LOSS_USD + 250.0)
killswitch_safe._record_market_data(datetime.now(UTC), {"DEMO": {"close": 99.0}}, {})

try:
    run_demo(killswitch_safe.submit_order_async("DEMO", 1))
except RiskLimitError as exc:
    print(f"daily_loss_block: {exc}")
else:
    raise AssertionError("Daily-loss breach did not block the order")

print(
    "post_breach_state: "
    f"kill_switch={killswitch_safe._state.kill_switch_activated} "
    f"daily_loss=${killswitch_safe._state.daily_loss:,.0f} "
    f"reason={killswitch_safe._state.kill_switch_reason!r}"
)
run_demo(killswitch_safe.disconnect())

2026-09-08 21:35:57,115 CRITICAL ml4t.live.safety: KILL SWITCH ACTIVATED: Daily loss $1,250.00 exceeds max $1,000.00


pre_drawdown_state: kill_switch=False
daily_loss_block: Daily loss $1,250.00 exceeds max $1,000.00
post_breach_state: kill_switch=True daily_loss=$1,250 reason='Daily loss $1,250.00 exceeds max $1,000.00'


### 2a. 从状态文件重建 `SafeBroker`

真实引擎重启会从其状态文件重新实例化 `SafeBroker`。锁存和
激活原因会持久化；下一笔订单会在到达 kill switch 下游的任何风险检查之前被拒绝。

In [7]:
killswitch_safe2 = SafeBroker(
    DemoBroker(account_value=100_000.0),
    LiveRiskConfig(
        execution_mode="paper",
        max_order_value=10_000.0,
        max_daily_loss=KILL_SWITCH_LOSS_USD,
        max_data_staleness_seconds=60.0,
        state_file=str(killswitch_state),
    ),
)
print(
    "after_restart_state: "
    f"kill_switch={killswitch_safe2._state.kill_switch_activated} "
    f"reason={killswitch_safe2._state.kill_switch_reason!r}"
)
run_demo(killswitch_safe2.connect())
killswitch_safe2._record_market_data(datetime.now(UTC), {"DEMO": {"close": 100.0}}, {})
try:
    run_demo(killswitch_safe2.submit_order_async("DEMO", 1))
except RiskLimitError as exc:
    print(f"post_restart_block: {exc}")
else:
    raise AssertionError("Persisted kill switch did not block the restarted broker")
run_demo(killswitch_safe2.disconnect())

# Reset for downstream demos
_cleanup_state_path(killswitch_state)

2026-09-08 21:35:57,184 WARNING ml4t.live.safety: Kill switch was previously activated: Daily loss $1,250.00 exceeds max $1,000.00


2026-09-08 21:35:57,185 WARNING ml4t.live.safety: SafeBroker reconciliation mismatch: missing_pending_orders=1


after_restart_state: kill_switch=True reason='Daily loss $1,250.00 exceeds max $1,000.00'
post_restart_block: Kill switch active: Daily loss $1,250.00 exceeds max $1,000.00


kill switch 强制满足 §25.7 的要求：它必须与被其控制的组件保持独立。由于锁存器存放在状态文件中，引擎崩溃并重连不会静默地重新启用交易；操作员必须显式清除锁存，引擎才会恢复运行。促使在 `12_ib_basket_rebalance_demo` 中引入启动对账的周五盘后堆积问题，是同一类故障发生在头寸维度而非盈亏（P&L）维度。

将开关锁存在持久化状态中，才使其在紧急情况下可用。非持久化的 kill switch 只是进程内存中的一个标志，而从混乱交易日中恢复时，是最不适合发现这种差异的时候。

## 3. 针对不一致状态文件的对账报告

`SafeBroker.connect()` 会将上一会话持久化的快照与 broker 的权威当前状态进行差异比较。下面的示例写入一个状态文件，声称策略在上一会话结束时持有 10 股 AAPL 多头，并有一张有效的 LIMIT 订单，然后连接到一个对两者都不知情的 broker。对账报告会列出每一处差异；生产启动器会拒绝启动新的交易周期，直到报告无差异或操作员显式清除持久化状态。

In [8]:
recon_state = _temp_state_path("nb13_recon_")
divergent = RiskState(
    date=datetime.now(UTC).date().isoformat(),
    persisted_positions={"AAPL": 10.0},
    persisted_pending_orders=[
        {
            "asset": "AAPL",
            "side": "buy",
            "quantity": 10.0,
            "order_type": "limit",
            "limit_price": 150.0,
        }
    ],
)
recon_state.write_text(json.dumps(divergent.to_dict(), indent=2))
recon_state.chmod(0o600)

recon_broker = DemoBroker(
    positions={
        "MSFT": Position(
            asset="MSFT",
            quantity=5,
            entry_price=410.0,
            entry_time=datetime.now(UTC),
        )
    }
)
recon_safe = SafeBroker(
    recon_broker,
    LiveRiskConfig(execution_mode="paper", state_file=str(recon_state)),
)
run_demo(recon_safe.connect())

report = recon_safe.reconciliation_report
print(f"clean: {report['clean']}")
print(f"missing_positions:    {report['missing_positions']}")
print(f"unexpected_positions: {report['unexpected_positions']}")
print(f"quantity_mismatches:  {report['quantity_mismatches']}")
print(f"missing_pending_orders:    {len(report['missing_pending_orders'])} order(s)")
print(f"unexpected_pending_orders: {len(report['unexpected_pending_orders'])} order(s)")
assert report["clean"] is False
assert report["missing_positions"] == {"AAPL": 10.0}
assert report["unexpected_positions"] == {"MSFT": 5.0}
assert len(report["missing_pending_orders"]) == 1

2026-09-08 21:35:57,292 WARNING ml4t.live.safety: SafeBroker reconciliation mismatch: missing_positions=['AAPL'], unexpected_positions=['MSFT'], missing_pending_orders=1


clean: False
missing_positions:    {'AAPL': 10.0}
unexpected_positions: {'MSFT': 5.0}
quantity_mismatches:  {}
missing_pending_orders:    1 order(s)
unexpected_pending_orders: 0 order(s)


报告会指出哪些头寸和订单不一致，以及不一致的方向。`missing_positions` 是持久化状态预期而 broker 不再报告的头寸，通常发生在手动平仓、盘后成交或隔夜公司行动之后。`unexpected_positions` 是 broker 处存在但持久化状态不知道的头寸，通常是最后一次持久化之后落地的成交，或非正常流程创建的头寸。

读取报告只完成一半任务；解决差异是另一半。操作员要么在 broker GUI 中调查该差异，并在弄清后重新运行；要么在已知该差异无害时，清除持久化状态并重连到干净的基线。

In [9]:
run_demo(recon_safe.disconnect())
_cleanup_state_path(recon_state)

# Clean baseline: persisted positions match the broker exactly
clean_state = _temp_state_path("nb13_recon_clean_")
matched_state = RiskState(
    date=datetime.now(UTC).date().isoformat(),
    persisted_positions={"MSFT": 5.0},
    persisted_pending_orders=[],
)
clean_state.write_text(json.dumps(matched_state.to_dict(), indent=2))
clean_state.chmod(0o600)

recon_broker_clean = DemoBroker(
    positions={
        "MSFT": Position(
            asset="MSFT",
            quantity=5,
            entry_price=410.0,
            entry_time=datetime.now(UTC),
        )
    }
)
recon_safe_clean = SafeBroker(
    recon_broker_clean,
    LiveRiskConfig(execution_mode="paper", state_file=str(clean_state)),
)
run_demo(recon_safe_clean.connect())
clean_report = recon_safe_clean.reconciliation_report
print(f"after_reset clean: {clean_report['clean']}")
assert clean_report["clean"] is True
run_demo(recon_safe_clean.disconnect())
_cleanup_state_path(clean_state)

after_reset clean: True


生产启动器应像对待失败的身份验证探测一样对待未通过（non-clean）的对账报告：拒绝启动并等待操作员处理。`12_ib_basket_rebalance_demo` 实现了该策略：如果报告未通过，IB basket 循环不会提交。

## 4. 引擎健康状态

`LiveEngine.runtime_status()` 返回一个 `dict`，其 `health` 字段将引擎和行情源状态归纳为少量操作员可读的类别：`stopped`、`waiting_for_data`、`ok`、`feed_silent`、`idle_market_closed` 和 `broker_disconnected`。下面的示例在合成行情源和空操作策略之上构建一个小型引擎，运行一个短的有限窗口，停止行情源，并报告健康状态转换。没有真实交易场所，没有 broker 连接，只有引擎自身的簿记。

玩具行情源未声明任何股票代码，因此引擎将其视为连续市场。这消除了对挂钟交易时段的依赖，并使健康状态序列在任意时刻都可复现。

In [10]:
@dataclass
class IdleStrategy(Strategy):
    """Strategy that ignores every bar, so only the health state is under observation."""

    bars_seen: int = 0

    def __post_init__(self) -> None:
        try:
            super().__init__()
        except TypeError:
            pass

    def on_data(self, *args: Any, **kwargs: Any) -> None:
        self.bars_seen += 1

    def on_start(self, *args: Any, **kwargs: Any) -> None:
        return None

    def on_end(self, *args: Any, **kwargs: Any) -> None:
        return None

#### 玩具行情数据源

一个最小的异步行情源，发出固定数量的 bar 后转为静默。将其与 `IdleStrategy` 配对，可让 LiveEngine 的健康状态转换显现（`waiting_for_data` → `ok` → `feed_silent`），而无需任何真实市场依赖。

In [11]:
@dataclass
class _ToyFeed:
    """Synthetic feed that emits one bar then goes silent until stopped."""

    initial_bars: int = 1
    start_delay_seconds: float = 0.2
    bar_interval_seconds: float = 0.1
    silent_after_initial: bool = True
    stopped: bool = field(default=False)

    async def start(self) -> None:
        self.stopped = False

    def stop(self) -> None:
        self.stopped = True

    def __aiter__(self):
        return self._stream()

    async def _stream(self):
        await asyncio.sleep(self.start_delay_seconds)
        for _ in range(self.initial_bars):
            if self.stopped:
                return
            yield (datetime.now(UTC), {"DEMO": {"close": 100.0}}, {})
            await asyncio.sleep(self.bar_interval_seconds)
        # Go silent: never yield more bars, just hold the iterator open
        while not self.stopped:
            await asyncio.sleep(0.1)

将行情源、策略和演示 broker 接入一个 `LiveEngine`，并设置较短的行情源静默超时，使看门狗在玩具行情源停止发出 bar 后迅速切换到 `feed_silent`。

In [12]:
broker_h = DemoBroker()
feed_h = _ToyFeed(initial_bars=2, start_delay_seconds=0.2, bar_interval_seconds=0.05)
strategy_h = IdleStrategy()
engine = LiveEngine(
    strategy=strategy_h,
    broker=broker_h,
    feed=feed_h,
    feed_silence_seconds=1.0,
    watchdog_poll_seconds=0.25,
)

`observe_engine` 在有限窗口内轮询 `runtime_status()`，并在每个预期健康状态转换发生时记录下来。

In [13]:
async def observe_engine() -> list[tuple[float, str]]:
    """Run the engine until each expected health transition occurs."""
    loop = asyncio.get_running_loop()
    started_at = loop.time()
    deadline = started_at + HEALTH_OBSERVATION_SECONDS
    transitions = [(0.0, engine.runtime_status()["health"])]

    async def wait_for_health(expected: str) -> None:
        while loop.time() < deadline:
            health = engine.runtime_status()["health"]
            if health == expected:
                transitions.append((round(loop.time() - started_at, 2), health))
                return
            await asyncio.sleep(0.01)
        observed = [health for _, health in transitions]
        raise AssertionError(f"expected health {expected!r}; observed {observed}")

    await engine.connect()
    run_task = asyncio.create_task(engine.run())
    for expected in ("waiting_for_data", "ok", "feed_silent"):
        await wait_for_health(expected)
    await engine.stop()
    run_task.cancel()
    try:
        await run_task
    except asyncio.CancelledError:
        pass
    transitions.append((round(loop.time() - started_at, 2), engine.runtime_status()["health"]))
    return transitions

In [14]:
timeline = run_demo(observe_engine())

In [15]:
last = None
transitions: list[dict] = []
for t, health in timeline:
    if health != last:
        transitions.append({"t_seconds": round(t, 2), "health": health})
        last = health
health_timeline = pl.DataFrame(transitions)
observed_health = health_timeline["health"].to_list()
assert observed_health == [
    "stopped",
    "waiting_for_data",
    "ok",
    "feed_silent",
    "stopped",
], observed_health
health_timeline

t_seconds,health
f64,str
0.0,"""stopped"""
0.01,"""waiting_for_data"""
0.21,"""ok"""
1.26,"""feed_silent"""
1.31,"""stopped"""


打印出的序列会对连续相同的健康状态去重，因此观察到的输出为 `stopped → waiting_for_data → ok → feed_silent → stopped`。短暂的初始延迟使等待状态可被观察到，而行情源的连续市场约定消除了对股票交易时段的依赖。这些转换都不需要真实交易场所。

`runtime_status()` 是看门狗或监督进程读取的内容。这些类别被刻意收窄，因为操作员会在压力下根据它们采取行动：`feed_silent` 所需的响应不同于 `broker_disconnected`，尽管从策略循环内部看，两者都像是“数据停止”。

In [16]:
assert not list(STATE_DIR.iterdir()), list(STATE_DIR.iterdir())
STATE_DIR.rmdir()
assert not STATE_DIR.exists(), STATE_DIR
print("Temporary state artifacts: cleaned")

Temporary state artifacts: cleaned


## 核心要点

1. **过期数据拒绝**可防止基于运行时已不再信任的价格进行交易。该阈值会在每个订单意图上按资产强制执行。
2. **单日亏损熔断开关**会锁存于持久化状态中；重建不会重置它，这正是它作为紧急控制手段有用的原因。
3. **启动对账**会暴露持久化快照与券商权威状态之间的每一处差异。生产启动器会拒绝在报告非干净状态时启动新周期。
4. **引擎健康状态**将运行时归纳为一小组面向操作员的词汇，监督进程可以直接消费这些状态。

**下一步**：`10_safety_risk_demo` 介绍可配置的风险面；
`12_ib_basket_rebalance_demo` 展示在实时 IB paper 篮子再平衡中的相同控制；§25.7 将抽象的预检要求与这里展示的强制运行时行为联系起来。